# 作业1：Data Types, Functions and Conditionals

- 学号：20234080121
- 姓名：曾凯锋
- 提交文件：`20234080121_曾凯锋_hw01.ipynb`


In [1]:
import os, uuid, hashlib, platform, getpass
# ===== 1. 填写学号  =====
STUDENT_ID = 20234080121  # 修改为自己的学号
# ===== 2. 打印  conda 环境  =====
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境 ] {CONDA_ENV}")
# ===== 3. 机器指纹（绑定当前电脑） =====
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
# ===== 4. 生成唯一  ID =====
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[ 学号 ]     {STUDENT_ID}")
print(f"[ 机器指纹 ] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")


[Conda 环境 ] base
[ 学号 ]     20234080121
[ 机器指纹 ] 88553ff75524cbdc
[RUN_ID]   20234080121-88553ff75524cbdc


## 热身：定义简单函数

1. `say_hi`：无参数，打印 `Hello, world!`。
2. `goat_pad`：接受一个字符串，前后各拼接 `goat` 后打印。
3. `print_n`：接受字符串 `s` 和整数 `n`，把 `s` 打印 `n` 次，每次单独一行。


In [2]:
def say_hi():
    print("Hello, world!")


def goat_pad(s):
    print("goat" + s + "goat")


def print_n(s, n):
    for _ in range(n):
        print(s)


In [3]:
say_hi()
goat_pad("max")
goat_pad("")  # 边界情况：空字符串
print_n("hello", 3)
print_n("goat", 0)  # n = 0 时不打印任何内容


Hello, world!
goatmaxgoat
goatgoat
hello
hello
hello


## 欧几里得算法

最大公约数（GCD）是能同时整除 m 和 n 的最大整数。
欧几里得算法基于辗转相除：`gcd(m, n) = gcd(n, m mod n)`，当 n 为 0 时 m 即为答案。
用循环实现：


In [4]:
def gcd(m, n):
    while n != 0:
        m, n = n, m % n
    return m


In [5]:
print(gcd(2018, 2019))  # 1，两个连续整数互质
print(gcd(1200, 300))   # 300
print(gcd(5040, 60))    # 60


1
300
60


In [6]:
# 第 3 问：负数输入时的行为
print(gcd(-1200, 300))
print(gcd(1200, -300))
print(gcd(-1200, -300))


300
-300
-300


**第 3 问回答：如果参数为负数，函数会输出什么？有意义吗？**

- `gcd(-1200, 300)` → `300`，`gcd(1200, -300)` → `-300`，`gcd(-1200, -300)` → `-300`。
- 由于 Python 的取余 `%` 运算结果的符号与除数一致，辗转相除后结果可能带上除数的符号。
- 因此函数返回值的**绝对值**始终是正确的最大公约数，但**符号可能为负**。
- 这在数学上仍然有意义：最大公约数通常定义为正整数，所以更严谨的写法是在返回时取绝对值 `return abs(m)`。本题假设输入为非负整数，因此保持原实现即可。


## 近似欧拉数 e

公式 (1)：$e = \sum_{k=0}^{\infty} \frac{1}{k!} = 1 + 1 + \frac{1}{2} + \frac{1}{6} + \cdots$（约定 $0! = 1$）

公式 (2)（伯努利极限）：$e = \lim_{x \to \infty} \left(1 + \frac{1}{x}\right)^x$


### 1. `euler_limit(n)`：用公式 (2) 取 x = n 近似 e


In [7]:
def euler_limit(n):
    return (1 + 1 / n) ** n


In [8]:
for n in (1, 10, 100, 10000):
    print(f"euler_limit({n:>5}) = {euler_limit(n)}")


euler_limit(    1) = 2.0
euler_limit(   10) = 2.5937424601000023
euler_limit(  100) = 2.7048138294215285
euler_limit(10000) = 2.7181459268249255


### 2. `euler_infinite_sum(n)`：公式 (1) 前 n 项之和（函数是 1 索引的）

注意：公式 (1) 从 k = 0 开始（0 索引），函数从 n = 1 开始（1 索引）。
因此 `euler_infinite_sum(1)` 返回 `1.0`（只用 k = 0 一项），`euler_infinite_sum(4)` 返回 `1 + 1 + 1/2 + 1/6 ≈ 2.6667`，`euler_infinite_sum(0)` 返回 `0`。

实现要点：不重复计算阶乘，用递推 `k! = k × (k−1)!` 逐步累加。


In [9]:
def euler_infinite_sum(n):
    if n == 0:
        return 0
    total = 0.0
    factorial = 1
    for k in range(n):  # 前 n 项即 k = 0 .. n-1
        if k > 0:
            factorial *= k
        total += 1 / factorial
    return total


In [10]:
print(euler_infinite_sum(0))  # 0
print(euler_infinite_sum(1))  # 1.0
print(euler_infinite_sum(4))  # 1 + 1 + 1/2 + 1/6 ≈ 2.6667
print(euler_infinite_sum(10))


0
1.0
2.6666666666666665
2.7182815255731922


### 3. `euler_approx(ε)`：用公式 (1) 的和逼近 e，误差小于 ε

提示要求使用 `while` 循环：不断增加项数 n，直到与真实值 `math.exp(1)` 的差小于 ε。


In [11]:
import math


def euler_approx(epsilon):
    n = 0
    approx = euler_infinite_sum(n)
    while abs(approx - math.exp(1)) >= epsilon:
        n += 1
        approx = euler_infinite_sum(n)
    return approx


In [12]:
approx = euler_approx(1e-10)
print(approx)
print(math.exp(1))                # e 的真实值（浮点精度内）
print(abs(approx - math.exp(1)))  # 误差 < 1e-10


2.7182818284467594
2.718281828459045
1.2285727990501982e-11


### 4. `print_euler_sum_table(n)`：打印 k = 1..n 时的部分和，每行一个


In [13]:
def print_euler_sum_table(n):
    for k in range(1, n + 1):
        print(k, euler_infinite_sum(k))


In [14]:
print_euler_sum_table(10)

1 1.0
2 2.0
3 2.5
4 2.6666666666666665
5 2.708333333333333
6 2.7166666666666663
7 2.7180555555555554
8 2.7182539682539684
9 2.71827876984127
10 2.7182815255731922


## 测试整数属性

正整数 n 是 2 的幂，当且仅当 n = 2^p（p 为某非负整数）。
思路：不断用 2 去除 n（只允许整除），如果最后能除到 1，则 n 是 2 的幂。
全程只使用除法（`//`）和模（`%`）运算，不使用 `math.sqrt`。


### 1. `is_power_of_2(n)`


In [15]:
def is_power_of_2(n):
    while n % 2 == 0:
        n //= 2
    return n == 1


In [16]:
print(is_power_of_2(1))     # True, 2^0
print(is_power_of_2(2))     # True
print(is_power_of_2(8))     # True, 2^3
print(is_power_of_2(12))    # False
print(is_power_of_2(1024))  # True, 2^10


True
True
True
False
True


### 2. `is_power(b, n)`：推广到任意底数 b

同样地不断用 b 整除 n，能除到 1 则说明 n = b^p。需要单独处理 b = 1 的边界（否则 `n % 1 == 0` 恒成立会死循环）：


In [17]:
def is_power(b, n):
    if b == 1:
        return n == 1
    while n % b == 0:
        n //= b
    return n == 1


In [18]:
print(is_power(2, 8))     # True,  2^3
print(is_power(3, 27))    # True,  3^3
print(is_power(5, 30))    # False
print(is_power(10, 1000)) # True,  10^3
print(is_power(7, 1))     # True,  b^0 = 1
print(is_power(1, 1))     # True
print(is_power(1, 5))     # False


True
True
False
True
True
True
False
